# Test du $\chi^2$



## 1 - Test d'ajustement à une loi 

On dispose d'un dé à 6 faces. 

On suppose que ce dé a été lancé $N=100$ fois et que l'on a obtenu: 

|Face 1|Face 2|Face 3 |Face 4|Face 5|Face 6|
|-|-------|-------|--------|--------|--------|
|17| 14   |   15|   18  |   16 | 20|

Est-il équilibré ? oui ils sont equilibres

Les hypothèses testées sont: 
* $H_0$: le dé est équilibré, 
* $H_1$: le dé n'est pas équilibré. 

C'est-à-dire: 
* $H_0$: on ne rejette pas
* $H_1$:  on  rejette 

Si $H_0$ est vraie, sur 100 essais on aurait le tableau suivant: 

|Face 1|Face 2|Face 3 |Face 4|Face 5|Face 6|
|-|-------|-------|--------|--------|--------|
|16.667| 16.667   |   16.667|   16.667  |   16.667 | 16.667|

**Exercice 1.1**: Calculer la statistique de décision 
$\chi^2_{obs} = \sum_{i=1}^k \frac{(n_i-t_i)^2}{t_i}$

In [6]:
import numpy as np

In [12]:
# Code 
import numpy as np

n_i = np.array([17, 14, 15, 18, 16, 20])   
t_i = np.array([16.667, 16.667, 16.667, 16.667, 16.667, 16.667])            
chi2_obs = np.sum((n_i - t_i)**2 / t_i)
print(chi2_obs)   # 1.4

1.399972040559189


On peut également le faire directement grâce au package **scipy**.

In [5]:
n_obs = [17,14,15,18,16,20]
n_theor = [100/6] * 6

from scipy import stats 

stats.chisquare(n_obs, n_theor)

Power_divergenceResult(statistic=1.4, pvalue=0.924313272801667)

Le test vous renvoie également la pvalue. 

**Exercice 1.2**: Pour $\alpha = 5\%$ quelle est la décision du test ? 

In [6]:
from scipy import stats

In [7]:
alpha = 0.05
res = stats.chisquare(n_obs, n_theor)

if res.pvalue < alpha:
    
    print("On rejette H0 : le dé n'est pas équilibré")
else:
    print("On ne rejette pas H0 : le dé est équilibré")

On ne rejette pas H0 : le dé est équilibré



**Exercice 1.3**: Charger le jeu de données *mais.csv* avec le package **pandas**. 

In [1]:
import pandas as pd

mais = pd.read_csv("data/mais.csv", sep=";")
mais.head()

,Individu,Hauteur,Masse,Nbgrains,Massegrains,Couleur,Germinationepi,Enracinement,Verse,Attaque,Parcelle,HauteurJ7,VerseTraitement,Nbjoursattaque,Censuredroite
0,1,NaN,NaN,NaN,NaN,NaN,NaN,Faible,NaN,Oui,Nord,171,NaN,NaN,NaN
1,2,199.0,1431.0,320.0,92.1,Rouge,Non,Moyen,Non,Non,Nord,196,Oui,NaN,NaN
2,3,205.0,1468.0,290.0,89.4,Jaune,Non,Moyen,Oui,Non,Nord,198,Oui,NaN,NaN
3,4,173.0,1398.0,147.0,42.6,Jaune,Non,Faible,Oui,Non,Nord,176,Oui,NaN,NaN
4,5,233.0,1622.0,138.0,43.2,Rouge,Non,Tresfort,Oui,Non,Nord,230,Oui,NaN,NaN


On veut tester si la répartition des couleurs dans le jeu de données *maïs* est uniforme. 

Pour ça, on crée un objet qui contient les effectifs observés.

In [10]:
n_obs = mais.Couleur.value_counts()
print(n_obs)

Jaune         48
Rouge         29
JauneRouge    22
Name: Couleur, dtype: int64


On crée l'objet qui contient les effectifs théoriques. 

In [5]:
# on obtient le nombre de couleur totale 
N_coul = sum(mais.Couleur.value_counts())

n_theor = [N_coul/3]*3

# on réalise le test
stats.chisquare(n_obs, n_theor)

Power_divergenceResult(statistic=10.969696969696969, pvalue=0.004149163692825389)

**Exercice 1.4**: Pour $\alpha = 5\%$ quelle est la décision du test ? Au seuil de confiance $97.5\%$ quelle est la décision du test ? 

In [14]:
n_obs = [17,14,15,18,16,20]
n_theor = [100/6] * 6
res = stats.chisquare(n_obs, n_theor)

for alpha in [0.05, 0.025]:
    if res.pvalue < alpha:
        print(f"alpha = {alpha} : on rejette H0")
    else:
        print(f"alpha = {alpha} : on ne rejette pas H0")

alpha = 0.05 : on ne rejette pas H0
alpha = 0.025 : on ne rejette pas H0


**Exercice 1.5**: On suppose que $30\%$ des parcelles sont orientées au nord, $20\%$ au sud, $17\%$ à l'est et $33\%$ à l'ouest.
Tester si la supposition ci-dessus peut être considérée comme
vraie sur l'ensemble de la population.

In [16]:
import numpy as np

In [17]:
# Code
n_obs = mais.Parcelle.value_counts()[["Nord", "Sud", "Est", "Ouest"]]
N = n_obs.sum()
p = np.array([0.30, 0.20, 0.17, 0.33])
n_theor = N * p

res = stats.chisquare(n_obs, n_theor)
print(res)

alpha = 0.05
if res.pvalue < alpha:
    print("On rejette H0")
else:
    print("On ne rejette pas H0")

Power_divergenceResult(statistic=22.76488413547237, pvalue=4.520595773369363e-05)
On rejette H0



Pour $\alpha = 5\%$ :

- si p-value $< 0.05$ : on rejette $H_0$, la supposition n'est pas valable pour l'ensemble de la population,


## 2 - Test d'indépendance 
L'objectif maintenance est de tester si deux variables qualitatives observées sur 1 échantillon sont indépendantes. 

Les hypothèses sont: 
* $H_0$ : les variables sont indépendantes. 
* $H_1$ : les variables ne sont pas indépendantes. 

Le test est alors basé sur un tableau de contingence. 

Prenons l'exemple d'un échantillon aléatoire. On s'intéresse à la couleur des cheveux en fonction de la couleur des yeux. 

||Blond|Chatain|Brun|Roux|
|-|-------|-------|--------|--------|
|Bleu| 25   |   9|   3  |   7 |
|Gris ou Vert| 13   |  17|  10  |   7 |
|Marron| 7  |  13|   8  |   5 |

**Exercice 2.1**: Peut-on appliquer le test du $\chi^2$ ? Détaillez votre réponse. 



Le test consiste à mesurer si la différence entre ce qu'on observe et ce qui devrait idéalement se passer en cas d'indépendance est statistiquement significatif. 

On construit un tableau d'éffectifs théoriques basé sur les marges du tableau de contingeance observé:
$t_{ij} = \frac{n_{i.} \times n_{.j}}{n}$
où 
* $t_{ij}$ : effectif théorique pour la ligne $i$ et la colonne $j$,
* $n_{i.}$: somme des effectifs observés pour la ligne $i$,
* $n_{.j}$: somme des effectifs observés pour la colonne $j$.

Ici le tableau des effectifs théoriques devient: 
||Blond|Chatain|Brun|Roux|
|-|-------|-------|--------|--------|
|Bleu| 15.96   |   13.84|   7.45 |   6.74 |
|Gris ou Vert| 17.06   |  14.78|  7.96  |   7.20 |
|Marron|  11.98  | 10.38|  5.59  |   5.06 |

La statistique de décision est : 
$\chi^2_{obs} = \sum_{i=1}^k \sum_{j=1}^c \frac{(n_{ij} \times t_{ij})^2}{t_{ij}}$. 

Sous $H_0$, $\chi^2_{obs} \sim \chi^2(d)$, avec $d = (k-1)(c-1)$. 


In [7]:
import numpy as np 

obs = np.array([[25,9,3,7], 
                [13,17,10,7], 
                [7,13,8,5]])

Ici, on peut directement tout faire avec la fonction *chi2_contingency()*:

In [8]:
chi2, p, dof, thq = stats.chi2_contingency(obs, correction = False)

print("La statistique du test est: ", chi2)

print("La p_value est: ", p)

print("Le nombre de degré de liberté est: ", dof)

print("La table des effectifs théoriques est: \n", thq)

La statistique du test est:  15.06663714950389
La p_value est:  0.01974471193518884
Le nombre de degré de liberté est:  6
La table des effectifs théoriques est: 
 [[15.96774194 13.83870968  7.4516129   6.74193548]
 [17.05645161 14.78225806  7.95967742  7.2016129 ]
 [11.97580645 10.37903226  5.58870968  5.05645161]]


*Remarque*: La correction ici fait référence à la correction de continuité de Yates qui doit être appliquée pour les tableaux de contingence $2 \times 2$.

**Exercice 2.2**: Coder une fonction *is_significant(tab, alpha)*, qui pour un tableau de contingence $tab$, et un risque $\alpha$,  effectue un test du $\chi^2$ et renvoie un booléen qui vaut True si le test est signifiant. Tester votre fonction.

In [9]:
# Code 

**Exercice 2.3**: Ajouter à votre fonction *is_significant(tab, alpha)* un message qui indique si la règle de Cochran est vérifier ou non dans le test. On pourra se servir d'une fonction annexe. 

In [10]:
# Code 

**Exercice 2.4**: A l'aide de vos deux fonctions précédemment crées, vérifier sur le jeu de données *mais.csv* si la parcelle et la couleur sont des variables aléatoires indépendantes. 
Pour cela, vous pouvez utiliser la fonction *crosstab* du package **pandas**: https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html. 

In [11]:
# Code 

Dans le cas où la règle de Cochran n'est pas vérifiée on peut réaliser un **test exact de Fisher**.

Pour l'appliquer en python il faut transformer le tableau des effectifs observés en array de numpy. 

## 3. A vous de jouer ! 

Ouvrez le jeu de données "titanic.csv". 
Dans ce jeu de données on s'intéresse à la survie des passagers en fonction de leur classe et leur sexe.  

Dans un premier temps, explorer le jeu de données afin d'étudier: 
* La présence (ou l'absence) de valeur manquante pour nos variables d'intérêts, 
* Le taux de survie, 
* La répartition des passagers selon les classes, 
* La répartition selon le sexe. 

A partir de ces premiers résultats, faîtes des commentaires. 

In [ ]:
titanic = pd.read_csv("data/titanic.csv", sep=",")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


La répartition Femmes / Hommes à bord suivait-elle la répartition naturelle d'une population standard ? 

In [ ]:
# Code 


Vous proposerez un moyen de visualiser la survie en fonction de la classe. Puis en fonction du sexe. 

In [ ]:
# Code 

A partir de ces graphiques, quel(s) hypothèse(s) pouvez-vous faire ? 

Vérifier vos hypothèses à l'aide d'un test adapté. 

In [ ]:
# Code 